<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_01_intraday_data_preparation/stage_01_intraday_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# stage_01_intraday_data_preparation



## Resumen

Esta notebook realiza la **preparación inicial del dataset intradía del MNQ (Micro E-mini Nasdaq 100)**.  
El objetivo es construir un dataset limpio, consistente y estructurado que servirá como base para la ingeniería de factores y el entrenamiento de modelos.

0. **Configuración del entorno**
   - Clonado del repositorio y montaje de Google Drive.
   - Instalación e importación de librerías necesarias.

1. **Fuente de datos**
   - Datos históricos intradía del MNQ (OHLCV, frecuencia de 1 minuto) exportados desde NinjaTrader.
   - Archivos originales en formato `.txt`, en zona horaria UTC.

2. **Generación del dataset**
   - Unificación de todos los archivos `.txt` en un único DataFrame.
   - Asignación de nombres de columnas: `open`, `high`, `low`, `close`, `volume`.
   - Conversión de la columna `datetime` a índice temporal.

3. **Filtrado**
   - Conserva solo **días hábiles bursátiles** (se eliminan fines de semana y feriados de mercado de EE.UU.).
   - Conversión de marcas de tiempo de **UTC → US/Eastern**.
   - Filtrado de **horario de mercado** (09:30–16:00) más pre-market (desde 08:30).

4. **Validación de registros diarios**
   - Verificación de que cada día contenga la cantidad esperada de registros minuto a minuto.
   - Detección y eliminación de días incompletos o con irregularidades.

5. **Chequeo de continuidad temporal**
   - Confirmación de que los datos intradía estén en intervalos consecutivos de 1 minuto, sin gaps.

6. **Dataset final**
   - Guardado del dataset limpio en formato `.parquet` dentro de Google Drive.

---

**Resultado:** Un dataset intradía del MNQ completamente limpio y estandarizado, listo para la ingeniería de factores y el modelado.

## 0. Configuración del Entorno

### 0.1. Clonado de repositorio / Acceso a Drive

In [ ]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

### 0.2. Instalación de librerías

In [ ]:
import sys
!{sys.executable} -m pip install -q pandas_market_calendars
print("✅ Librería instalada: pandas_market_calendars")

✅ Librería instalada: pandas_market_calendars


### 0.3. Importación de librerías

In [ ]:
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import pandas as pd
from tabulate import tabulate

# Calendario de mercados
import pandas_market_calendars as mcal
import pandas as pd
import requests
from io import StringIO


### 0.4. Acceso a archivos locales/remotos

In [ ]:
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd                 # procesamiento de datos
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
RAW_DIR = Path(os.environ.get("RAW_DIR", "data/raw/mnq_raw.parquet"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/processed/mnq_intraday.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/dataset_prep_summary.json"))

In [ ]:
# PARA EL NOTEBOOK:
RAW_DIR = DRIVE_DIR / RAW_DIR
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

## 1. Carga del dataset crudo `mnq_raw`

In [ ]:
def load_raw_dataset():
    os.path.exists(RAW_DIR)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_raw = pd.read_parquet(RAW_DIR)
    return mnq_raw

In [ ]:
mnq_raw = load_raw_dataset()

Archivo encontrado en disco. Cargando dataset local...


In [ ]:
def count_total_days(df: pd.DataFrame) -> int:
    """
    Cuenta la cantidad total de días distintos presentes en un DataFrame
    con índice de tipo DatetimeIndex.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame indexado por datetime.

    Retorna
    -------
    int
        Número total de días distintos en el dataset.
    """
    # Extraer la fecha (YYYY-MM-DD) del índice y contar valores únicos
    total_days = df.index.normalize().nunique()

    return int(total_days)

In [ ]:
total_days_raw = count_total_days(mnq_raw)
print(f"Total days (RAW): {total_days_raw}")

Total days (RAW): 1756


## 2. Filtrado de días no hábiles y horario bursátil

### 2.1. Filtrado de fines de semana y feriados bursátiles estadounidenses

Es necesario filtrar del conjunto de datos aquellas filas correspondientes a sábados, domingos y feriados bursátiles. Para ello, se utilizará la librería pandas_market_calendars, que permite identificar los días hábiles de operación según el calendario oficial del NASDAQ.

La función implementada filtra un DataFrame con índice de tipo DatetimeIndex, conservando únicamente aquellas filas cuya fecha coincida con un día hábil del mercado. La marca temporal completa (fecha y hora) se mantiene sin modificaciones.

In [ ]:
def filter_nasdaq_trading_days(df):
    """
    Filtra un DataFrame para conservar únicamente los días hábiles
    de negociación del mercado NASDAQ, en función de su DatetimeIndex.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame indexado por fechas y horas (DatetimeIndex).

    Retorna
    -------
    pandas.DataFrame
        DataFrame que contiene solo las filas correspondientes
        a días oficiales de trading del NASDAQ.
    """
    # Inicializar el calendario oficial del mercado NASDAQ
    nasdaq_calendar = mcal.get_calendar("NASDAQ")

    # Determinar el rango de fechas a partir del índice del DataFrame
    start_date = df.index.min().date()
    end_date = df.index.max().date()

    # Obtener el cronograma oficial de días hábiles del NASDAQ
    trading_days = nasdaq_calendar.schedule(
        start_date=start_date,
        end_date=end_date
    ).index.date

    # Filtrar el DataFrame manteniendo solo fechas válidas de trading
    filtered_df = df[df.index.normalize().isin(trading_days)]

    return filtered_df

In [ ]:
mnq_intraday = filter_nasdaq_trading_days (mnq_raw)

In [ ]:
trading_days = count_total_days(mnq_intraday)
print(f"Trading days: {trading_days}")

Trading days: 1370


### 2.2. Filtrado de horario de operación de mercado de New York (09:30 a 16:00) con pre mercado, desde las 06:30

Dado que los timestamps del índice (DatetimeIndex) provienen de archivos .txt sin información de zona horaria, es necesario indicar explícitamente a pandas que dichos valores están en formato UTC.

Una vez establecido el timezone, se procede a convertir los timestamps desde UTC a la hora local del mercado estadounidense (zona US/Eastern), correspondiente a los horarios de operación del NASDAQ/NYSE. Esta conversión se realiza teniendo en cuenta automáticamente los ajustes por horario de verano o invierno.

In [ ]:
def configure_timezone(df, from_tz="UTC", to_tz="America/New_York"):
    """
    Asegura que el índice del DataFrame tenga definida la zona horaria
    `from_tz` y luego lo convierte a la zona horaria `to_tz`.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame con índice de tipo DatetimeIndex.
    from_tz : str, opcional
        Zona horaria de origen a asignar si el índice no tiene tz (por defecto "UTC").
    to_tz : str, opcional
        Zona horaria destino a la que se convertirá el índice
        (por defecto "America/New_York").

    Retorna
    -------
    pandas.DataFrame
        DataFrame con el índice correctamente localizado y convertido.
    """
    # Verificar si el índice no tiene información de zona horaria
    if df.index.tz is None:
        # Asignar la zona horaria de origen sin modificar los timestamps
        df.index = df.index.tz_localize(from_tz)

    # Convertir el índice a la zona horaria destino
    df.index = df.index.tz_convert(to_tz)

    return df

In [ ]:
mnq_intraday = configure_timezone(mnq_intraday)

In [ ]:
mnq_intraday

,open,high,low,close,volume
datetime,,,,,
2019-12-22 22:01:00-05:00,8718.50,8718.75,8718.50,8718.50,9
2019-12-22 22:02:00-05:00,8718.25,8718.25,8718.00,8718.25,14
2019-12-22 22:03:00-05:00,8718.25,8718.50,8718.00,8718.25,74
2019-12-22 22:04:00-05:00,8718.25,8719.00,8718.25,8718.50,10
2019-12-22 22:05:00-05:00,8718.50,8719.00,8718.50,8719.00,6
...,...,...,...,...,...
2025-06-13 17:00:00-04:00,21651.25,21659.50,21649.75,21654.00,610
2025-06-15 20:00:00-04:00,21686.50,21691.75,21686.00,21687.25,103
2025-06-15 20:01:00-04:00,21687.25,21694.00,21684.50,21688.00,266


La siguiente función selecciona únicamente las muestras que se encuentran dentro del horario regular de operación bursátil del NASDAQ.

Filtra un DataFrame cuyo índice es de tipo DatetimeIndex, conservando solo aquellas filas cuya marca temporal se encuentre entre las 09:30 y 16:00 horas (US/Eastern), correspondientes al horario de negociación estándar en días hábiles de mercado.

Particularmente, decido agregar una hora de pre mercado, desde las 06:30AM.

In [ ]:
def filter_nasdaq_trading_hours(df, start_time: str, end_time: str):
    """
    Filtra un DataFrame para conservar únicamente las filas que se
    encuentren dentro del horario de negociación del NASDAQ.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame indexado por DatetimeIndex.
    start_time : str
        Hora de inicio del mercado (por ejemplo, "08:00:00").
    end_time : str
        Hora de cierre del mercado (por ejemplo, "16:00:00").

    Retorna
    -------
    pandas.DataFrame
        DataFrame filtrado dentro del rango horario especificado.
    """
    # Filtrar filas que estén dentro del rango horario de trading
    filtered_df = df.between_time(start_time, end_time)

    return filtered_df

In [ ]:
mnq_intraday = filter_nasdaq_trading_hours(mnq_intraday, '06:30:00', '16:00:00' )

In [ ]:
trading_session_days = count_total_days(mnq_intraday)
print(f"Trading session days (06:30 to 16:00): {trading_session_days }")

Trading session days (06:30 to 16:00): 1367


## 3. Análisis de registros diarios

Es necesario verificar que todos los días del conjunto de datos contengan la misma cantidad de registros y que estos sean consecutivos, es decir, que no falte ningún minuto dentro de cada jornada.

La función analizar_registros_por_dia permite realizar este control sobre un DataFrame con índice de tipo datetime. La función contabiliza la cantidad de registros por día e imprime una tabla resumen que indica cuántos días presentan una determinada cantidad de registros. Esto resulta útil para identificar inconsistencias, como días incompletos o interrupciones en la frecuencia temporal esperada.

In [ ]:
def analyze_daily_record_counts(df: pd.DataFrame) -> tuple[pd.Series, int]:
    """
    Analiza la cantidad de registros por día en un DataFrame con índice datetime.

    Imprime:
    - La cantidad de registros correspondiente a un día completo.
    - El número y porcentaje de días con menos registros que dicho valor.

    Retorna:
    -------
    tuple[pandas.Series, int]
        - Serie con el conteo de registros por día.
        - Cantidad de registros correspondiente a un día completo
          (valor más frecuente).
    """
    # Contar la cantidad de registros por día
    daily_counts = df.groupby(df.index.date).size()

    # Determinar el número de registros de un día completo (moda)
    full_day_records = daily_counts.mode().iloc[0]
    print(f"Cantidad de registros en un día completo: {full_day_records}")

    # Calcular el porcentaje de días con registros incompletos
    total_days = len(daily_counts)
    incomplete_days = (daily_counts < full_day_records).sum()
    incomplete_percentage = (incomplete_days / total_days) * 100

    print(
        f"Días con menos de {full_day_records} registros: "
        f"{incomplete_days} de {total_days} ({incomplete_percentage:.2f}%)"
    )

    return daily_counts, full_day_records

In [ ]:
#resumen, registros_dia_completo = analyze_daily_record_counts(mnq_intraday)
daily_record_counts, full_day_record_count = analyze_daily_record_counts(mnq_intraday)

Cantidad de registros en un día completo: 571
Días con menos de 571 registros: 64 de 1367 (4.68%)


Como podemos observar en la tabla, la gran mayoría de días tienen `571` muestras. Y representan más del 95% del total de los datos.


### 3.1. Filtrado de días incompletos

La siguiente función encuentra los indices de las fechas con registros incompletos:

In [ ]:
def find_incomplete_trading_dates(
    df: pd.DataFrame,
    expected_records: int,
    gap_minutes: int = 1,
) -> list[pd.Timestamp]:
    """
    Identifica los días que presentan menos registros de los esperados
    o irregularidades en la secuencia temporal de los datos.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame con índice de tipo DatetimeIndex.
    expected_records : int
        Cantidad esperada de registros por día.
    gap_minutes : int, opcional
        Intervalo esperado entre registros consecutivos, en minutos
        (por defecto 1 minuto).

    Retorna
    -------
    list[pandas.Timestamp]
        Lista de fechas que presentan registros incompletos
        o irregularidades temporales.
    """
    df = df.copy()

    # Calcular la diferencia temporal entre registros consecutivos
    df["time_diff"] = df.index.to_series().diff()
    expected_time_diff = pd.Timedelta(minutes=gap_minutes)

    # Conteo de registros por día
    daily_counts = df.groupby(df.index.date).size()

    problematic_dates = []

    # Analizar cada día de forma independiente
    for date, group in df.groupby(df.index.date):
        time_diffs = group["time_diff"].iloc[1:]
        has_irregular_gaps = (time_diffs != expected_time_diff).any()
        record_count = daily_counts[date]

        if record_count < expected_records or has_irregular_gaps:
            problematic_dates.append(date)

    return problematic_dates

Elimino las fechas con registros incompletos:

In [ ]:
def remove_incomplete_trading_days(
    df: pd.DataFrame,
    expected_records: int,
) -> pd.DataFrame:
    """
    Elimina del DataFrame los días que presentan registros incompletos
    o irregularidades temporales, y vuelve a analizar los registros diarios.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame con índice de tipo DatetimeIndex.
    expected_records : int
        Cantidad esperada de registros por día.

    Retorna
    -------
    pandas.DataFrame
        DataFrame filtrado, conteniendo únicamente días completos
        y sin irregularidades temporales.
    """
    # Identificar fechas problemáticas (días incompletos o con gaps)
    problematic_dates = find_incomplete_trading_dates(
        df=df,
        expected_records=expected_records,
    )

    # Filtrar el DataFrame eliminando las fechas con problemas
    cleaned_df = df[
        ~df.index.to_series().dt.date.isin(problematic_dates)
    ]

    # Reanalizar la cantidad de registros por día tras la limpieza
    analyze_daily_record_counts(cleaned_df)

    return cleaned_df

In [ ]:
#df_mnq = eliminar_fechas_incompletas(df_mnq, registros_dia_completo)
mnq_intraday = remove_incomplete_trading_days(
    df = mnq_intraday,
    expected_records = full_day_record_count,
)

Cantidad de registros en un día completo: 571
Días con menos de 571 registros: 0 de 1303 (0.00%)


np.int64(571)

In [ ]:
trading_session_complete_days = count_total_days(mnq_intraday)
print(f"Trading session complete days : {trading_session_complete_days }")

Trading session complete days : 1303


## 4. Verificación de continuidad temporal minuto a minuto

Es necesario verificar que los registros correspondientes a un mismo día estén dispuestos de forma consecutiva, con una separación exacta de un minuto entre cada muestra.

In [ ]:
def detect_time_gaps(
    df: pd.DataFrame,
    gap_minutes: int = 1,
) -> list[pd.DatetimeIndex]:
    """
    Verifica la existencia de saltos temporales mayores al intervalo esperado
    entre registros consecutivos dentro de cada día.

    Se omite el primer registro de cada jornada, ya que no tiene referencia previa.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame con índice de tipo DatetimeIndex.
    gap_minutes : int, opcional
        Intervalo temporal esperado entre registros consecutivos, en minutos
        (por defecto 1 minuto).

    Retorna
    -------
    list[pandas.DatetimeIndex]
        Lista de índices donde se detectaron diferencias temporales
        distintas al intervalo esperado.
    """
    df = df.copy()

    # Calcular diferencias temporales entre registros consecutivos
    df["time_diff"] = df.index.to_series().diff()
    expected_time_diff = pd.Timedelta(minutes=gap_minutes)

    problematic_indices = []

    # Analizar cada día de manera independiente
    for date, group in df.groupby(df.index.date):
        time_diffs = group["time_diff"].iloc[1:]
        irregular_indices = time_diffs[time_diffs != expected_time_diff].index

        if not irregular_indices.empty:
            problematic_indices.append(irregular_indices)

    if problematic_indices:
        print(
            f"Se encontraron problemas en {len(problematic_indices)} "
            "bloques diarios con diferencias temporales irregulares.\n"
        )

        # Conteo de registros por día
        daily_counts = df.groupby(df.index.date).size()

        for indices in problematic_indices:
            idx = indices[0]
            time_diff = df.loc[idx, "time_diff"]
            date = idx.date()
            record_count = daily_counts[date]

            print(
                f"\t{idx} -> Diferencia: {time_diff} "
                f"| # Registros del día: {record_count}"
            )
    else:
        print(
            "No se encontraron problemas: todas las muestras "
            "son consecutivas minuto a minuto."
        )

    return problematic_indices

In [ ]:
detect_time_gaps(mnq_intraday)

No se encontraron problemas: todas las muestras son consecutivas minuto a minuto.


[]

## 5. Búsqueda de NaNs


In [ ]:
# Verificar si existen NaNs en el dataset
has_nans = mnq_intraday.isna().any().any()

# Cantidad total de NaNs
total_nans = int(mnq_intraday.isna().sum().sum())

# Cantidad de NaNs por columna
nans_by_column = mnq_intraday.isna().sum()

print("=== NaN Check: mnq_intraday ===")
print(f"¿Existen NaNs en el dataset?: {has_nans}")
print(f"Cantidad total de NaNs: {total_nans}\n")

print("NaNs por columna:")
#display(nans_by_column.to_frame(name="NaN count"))
print(nans_by_column)


=== NaN Check: mnq_intraday ===
¿Existen NaNs en el dataset?: False
Cantidad total de NaNs: 0

NaNs por columna:
open      0
high      0
low       0
close     0
volume    0
dtype: int64


## 6. Búsqueda de Hora inicial y final

In [ ]:
def get_trading_time_range(df: pd.DataFrame) -> tuple[str, str]:
    """
    Obtiene el horario inicial y final presentes en un DataFrame intradía
    con índice DatetimeIndex.

    Retorna
    -------
    tuple[str, str]
        Hora inicial y hora final en formato HH:MM:SS.
    """
    times = df.index.time

    start_time = min(times).strftime("%H:%M:%S")
    end_time = max(times).strftime("%H:%M:%S")

    return start_time, end_time

In [ ]:
start_time, end_time = get_trading_time_range(mnq_intraday)

print("Horario del dataset mnq_intraday")
print(f"Inicio: {start_time}")
print(f"Fin:    {end_time}")

Horario del dataset mnq_intraday
Inicio: 06:30:00
Fin:    16:00:00


## 5. Guardado de dataset final



Se guarda un dataset compuesto por 1198 días, cada uno con 511 registros correspondientes a minutos consecutivos.

El conjunto de datos incluye únicamente días hábiles de operación bursátil, y abarca el intervalo horario comprendido entre las 07:30 y las 16:00 horas (US/Eastern).

In [ ]:
# Ruta del dataset en Drive
#mnq_intraday_data_file = f"{drive_path}/1_mnq_dataset_preparation/mnq_intraday_data.parquet"

# Verificar si el archivo ya existe
if os.path.exists(OUT_PARQUET):
    print(f"Archivo encontrado en disco: {OUT_PARQUET}")
    mnq_intraday = pd.read_parquet(OUT_PARQUET)
    print("Dataset cargado desde Drive.")
else:
    print("No se encontró el archivo en Drive. Guardando nuevo dataset...")
    os.makedirs(os.path.dirname(OUT_PARQUET), exist_ok=True)
    mnq_intraday.to_parquet(OUT_PARQUET, index=True)
    print(f"Dataset guardado en: {OUT_PARQUET}")

No se encontró el archivo en Drive. Guardando nuevo dataset...
Dataset guardado en: /content/drive/MyDrive/neural_profit/data/processed/mnq_intraday.parquet


## 6. Summary, params y MLFlow


In [ ]:
#print(f"Total days mnq_raw (input) : {total_days_raw}")
#print(f"Trading days: {trading_days}")
#print(f"Trading session days (06:30 to 16:00): {trading_session_days }")
#print(f"Total days mnq_intraday (output) : {trading_session_complete_days }")
#print(f"Cantidad de registros por día : {full_day_record_count }")
#print(f"Cantidad total de NaNs: {total_nans}")
#print(f"Hora de inicio: {start_time}")
#print(f"Hora de final: {end_time}")


Total days mnq_raw (input) : 1756
Total days mnq_intraday (output) : 1303
Cantidad de registros por día : 571
Cantidad total de NaNs: 0
Hora de inicio: 06:30:00
Hora de final: 16:00:00


In [ ]:
dataset_prep_summary = {
    "input": {
        "total_days_raw": int(total_days_raw),
    },
    "output": {
        "trading_session_complete_days": int(trading_session_complete_days),
        "expected_records_per_day": int(full_day_record_count),
        "trading_time_range": {
            "start_time": start_time,
            "end_time": end_time,
        },
    },
    "quality_checks": {
        "total_nans": int(total_nans),
        "discarded_days": int(total_days_raw - trading_session_complete_days),
        "discarded_days_pct": round(
            (total_days_raw - trading_session_complete_days) / total_days_raw * 100, 2
        ),
    },
}

### Guardar reports/dataset_prep_summary.jso

In [ ]:
from pathlib import Path
import json

# ------------------------------------------------------------
# Guardado del summary del stage_01 en formato JSON
# ------------------------------------------------------------

# Definir la ruta del archivo de resumen del dataset
summary_path = Path("reports/dataset_prep_summary.json")

# Crear la carpeta 'reports/' si no existe
# (parents=True permite crear toda la jerarquía necesaria)
summary_path.parent.mkdir(parents=True, exist_ok=True)

# Guardar el diccionario dataset_prep_summary como archivo JSON
# - indent=2: mejora la legibilidad del archivo
# - ensure_ascii=False: permite caracteres UTF-8 (acentos, etc.)
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(dataset_prep_summary, f, indent=2, ensure_ascii=False)

### Logging limpio en MLflow (stage_01)

In [ ]:
import mlflow

# ------------------------------------------------------------
# Registro de parámetros (configuración del dataset)
# ------------------------------------------------------------

# Mercado al que pertenecen los datos
mlflow.log_param("market", "NASDAQ")

# Horario efectivo de trading presente en el dataset
mlflow.log_param("trading_start_time", start_time)
mlflow.log_param("trading_end_time", end_time)

# ------------------------------------------------------------
# Registro de métricas (calidad y características del dataset)
# ------------------------------------------------------------

# Cantidad total de días presentes en el dataset original (mnq_raw)
mlflow.log_metric("total_days_raw", total_days_raw)

# Cantidad de días válidos luego del filtrado por trading session
mlflow.log_metric("trading_days", trading_session_complete_days)

# Cantidad de días descartados durante la preparación del dataset
mlflow.log_metric(
    "discarded_days",
    total_days_raw - trading_session_complete_days,
)

# Porcentaje de días descartados respecto del total original
mlflow.log_metric(
    "discarded_days_pct",
    dataset_prep_summary["quality_checks"]["discarded_days_pct"],
)

# Cantidad esperada de registros por día completo de trading
mlflow.log_metric("records_per_day", full_day_record_count)

# Cantidad total de valores NaN detectados en el dataset final
mlflow.log_metric("total_nans", total_nans)

# ------------------------------------------------------------
# Registro de artefactos
# ------------------------------------------------------------

# Guardar el summary del stage_01 como artefacto de MLflow
# (útil para auditoría, reproducibilidad y trazabilidad)
mlflow.log_artifact(str(summary_path))

ModuleNotFoundError: No module named 'mlflow'

## 7. Alineación con libro ML


**1. Estructura general del dataset**

- **Formato**: parquet.  
- **Índice temporal**: `datetime` con timezone.  
- **Columnas**: `open`, `high`, `low`, `close`, `volume` (OHLCV).  
- **Tamaño**: ~744.000 registros, consistente con un dataset intradía multianual.

    **Conclusión**: estructura sólida y estándar para modelado ML intradía.

**2. Orden temporal y continuidad**

- Los timestamps se encuentran **estrictamente ordenados**.  
- No se detectan **duplicados por minuto**.  
- El índice temporal es **monótono creciente**.

    **Conclusión**: no existen problemas de orden ni desalineación temporal.

**3. Separación por jornada**

- Los datos están **segmentados por jornada bursátil**.  
- No se mezclan observaciones entre días.  
- Cada jornada respeta una **ventana intradía consistente**.

    Este punto es crítico para el modelado intradía y se encuentra correctamente implementado.

**4. Horario de sesión**

- El dataset base cubre el intervalo **06:30–16:00 (ET)**.  
- Se aplica un **filtrado posterior** para definir ventanas específicas de uso  
  (por ejemplo, *gestation* / *trading window*).

**5. Coherencia OHLCV**

- Se cumple la relación `high ≥ open/close ≥ low`.  
- Los volúmenes son **no negativos**.  
- No se observan valores aberrantes evidentes.

    **Conclusión**: datos de mercado consistentes y sanos.

**6. Base para targets H = 60 / H = 90**

- El dataset presenta **profundidad intradía suficiente** para construir targets a horizontes H = 60 y H = 90 minutos.  
- Los últimos minutos de cada jornada deben descartarse al construir los targets, de acuerdo con la definición temporal del problema.

---

**Diagnóstico global**

La preparación de los datos es **correcta, limpia y metodológicamente sólida**.  
No se identifican señales de *look-ahead bias*, mezcla de jornadas ni inconsistencias estructurales.
